# In-House Layout Clustering Pipeline

**목적**: Layout feature 126개 × 280,000 rows 데이터에서  
- CD (nm) 산포를 최소화하는 클러스터 도출  
- SHAP 기반 feature 중요도 분석 및 선택  
- **Cost function**: `α × mean(4σ range %) + β × max(4σ range %)` 최소화  

**4σ range % 정의**:
```
1. CD_norm   = CD / median_CD          # median_CD로 정규화 → 정규화 후 median = 1
2. 4σ_upper  = mean(CD_norm) + 4·σ(CD_norm)
3. 4σ_lower  = mean(CD_norm) - 4·σ(CD_norm)
4. 4σ range % = (4σ_upper - 4σ_lower) × 100
              = 8·σ(CD_norm) × 100
```

- **제약**: 클러스터당 최소 샘플 수 ≥ MIN_CLUSTER_COUNT  
- **median alignment 효과 분석**: 클러스터별 CD median을 전체 median으로 보정 후 4σ range % 개선량 확인  

**알고리즘 3종 비교**:  
1. Decision Tree Clustering  
2. K-Means (SHAP feature 기반)  
3. **Autoencoder + K-Means** (DL 잠재 표현 기반)

## 0. 설정 (Configuration)

In [ ]:
# ============================================================
#  GLOBAL CONFIG — 이 셀만 수정하면 전체 파이프라인 제어 가능
# ============================================================

DATA_PATH  = None           # None → 합성 데이터 자동 생성
# DATA_PATH = 'data/layout_features.csv'

CD_COLUMN  = 'CD_nm'        # CD 컬럼명
N_ROWS     = 280_000
N_FEAT     = 126

# ── SHAP ────────────────────────────────────────────────────
SHAP_TOP_K       = 20
SHAP_SAMPLE_N    = 10_000
XGB_N_ESTIMATORS = 200
XGB_MAX_DEPTH    = 6

# ── 클러스터 제약 ───────────────────────────────────────────
MIN_CLUSTER_COUNT = 100     # 클러스터 최소 샘플 수

# ── Cost function 가중치 (4σ% 기준) ─────────────────────────
ALPHA = 1.0   # mean(4σ%) 가중치
BETA  = 0.5   # max(4σ%) 가중치

# ── Grid search 범위 ─────────────────────────────────────────
DT_MAX_LEAF_RANGE = list(range(5, 81, 5))
KMEANS_K_RANGE    = list(range(5, 51, 5))
AE_K_RANGE        = list(range(5, 51, 5))   # AE 잠재공간 K-Means

# ── Autoencoder 설정 ─────────────────────────────────────────
AE_LATENT_DIM  = 16
AE_EPOCHS      = 40
AE_BATCH_SIZE  = 2048
AE_LR          = 1e-3
AE_PATIENCE    = 5          # early stopping patience

# ── 기타 ─────────────────────────────────────────────────────
RANDOM_STATE = 42
OUTPUT_DIR   = 'output'

import os, warnings
warnings.filterwarnings('ignore')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Config loaded.')
print(f'  CD column        : {CD_COLUMN}')
print(f'  4σ% cost         : α={ALPHA}·mean + β={BETA}·max')
print(f'  Min cluster cnt  : {MIN_CLUSTER_COUNT}')
print(f'  SHAP Top-K       : {SHAP_TOP_K}')
print(f'  AE latent dim    : {AE_LATENT_DIM}')

## 1. 라이브러리 임포트

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from tqdm.auto import tqdm
from pathlib import Path
from collections import Counter
import json, joblib

from sklearn.preprocessing import RobustScaler
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

import xgboost as xgb
import shap

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

shap.initjs()
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch device: {DEVICE}')
print('Libraries loaded.')

## 2. 데이터 로딩 & EDA

In [ ]:
def generate_synthetic_data(n_rows, n_feat, cd_col, seed):
    rng = np.random.default_rng(seed)
    group = rng.choice([0,1,2,3,4], size=n_rows, p=[0.35,0.25,0.20,0.12,0.08])
    X = rng.standard_normal((n_rows, n_feat)).astype(np.float32)
    for g in range(5):
        mask = group == g
        X[mask] = X[mask] * rng.uniform(0.5,1.5,n_feat) + rng.uniform(-2,2,n_feat)
    X[rng.random((n_rows, n_feat)) < 0.01] = np.nan
    feat_names = [f'feat_{i:03d}' for i in range(n_feat)]
    df = pd.DataFrame(X, columns=feat_names)
    coef = rng.uniform(-1, 1, n_feat)
    cd_base = df.fillna(0).values @ coef * 0.3
    group_offset = np.array([50, 70, 90, 110, 130])[group]
    noise_std    = np.array([3,  5,  4,  6,   8  ])[group]
    df[cd_col] = (cd_base + group_offset + rng.normal(0, noise_std, n_rows)).astype(np.float32)
    df['_true_group'] = group
    return df

if DATA_PATH is None:
    print(f'합성 데이터 생성 중 ({N_ROWS:,} × {N_FEAT})...')
    df_raw = generate_synthetic_data(N_ROWS, N_FEAT, CD_COLUMN, RANDOM_STATE)
else:
    df_raw = pd.read_csv(DATA_PATH)

print(f'Shape: {df_raw.shape}')
df_raw.head(3)

In [ ]:
feat_cols = [c for c in df_raw.columns if c not in [CD_COLUMN, '_true_group']]

# ── CD EDA용 4σ 경계 계산 (전처리 셀 이전이므로 여기서 직접 계산) ──
cd_eda      = df_raw[CD_COLUMN].dropna().values
med_eda     = float(np.median(cd_eda))
cd_norm_eda = cd_eda / med_eda                  # 정규화 (median = 1)
mean_n_eda  = cd_norm_eda.mean()
sig_n_eda   = cd_norm_eda.std()
upper_norm  = mean_n_eda + 4.0 * sig_n_eda      # 4σ upper (normalized)
lower_norm  = mean_n_eda - 4.0 * sig_n_eda      # 4σ lower (normalized)
upper_cd    = upper_norm * med_eda               # nm 단위로 역변환
lower_cd    = lower_norm * med_eda               # nm 단위로 역변환
range_pct   = (upper_norm - lower_norm) * 100    # 4σ range %

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── CD 분포 + 4σ 경계 ───────────────────────────────────────
axes[0].hist(cd_eda, bins=120, color='steelblue', edgecolor='none', alpha=0.75, density=True)

# 4σ upper / lower 수직선
axes[0].axvline(upper_cd, color='red',    ls='--', lw=1.8,
                label=f'4σ upper = {upper_cd:.2f} nm')
axes[0].axvline(lower_cd, color='orange', ls='--', lw=1.8,
                label=f'4σ lower = {lower_cd:.2f} nm')
axes[0].axvline(med_eda,  color='navy',   ls=':',  lw=1.5,
                label=f'median = {med_eda:.2f} nm')

# 4σ range 영역 shading
ymax = axes[0].get_ylim()[1] if axes[0].get_ylim()[1] > 0 else 1
axes[0].axvspan(lower_cd, upper_cd, alpha=0.08, color='red')

# 4σ range % 어노테이션
axes[0].annotate(
    f'4σ range = {range_pct:.3f} %\n'
    f'(upper−lower) × 100\n'
    f'on CD_norm = CD / {med_eda:.2f}',
    xy=(upper_cd, 0), xycoords='data',
    xytext=(0.97, 0.95), textcoords='axes fraction',
    ha='right', va='top', fontsize=9,
    bbox=dict(boxstyle='round,pad=0.4', fc='lightyellow', ec='gray', alpha=0.9),
    arrowprops=dict(arrowstyle='->', color='red', lw=1.2),
)

axes[0].set_title('CD (nm) Distribution with 4σ Bounds', fontweight='bold')
axes[0].set_xlabel('CD (nm)'); axes[0].set_ylabel('Density')
axes[0].legend(fontsize=8)

# ── 결측치 비율 ─────────────────────────────────────────────
missing_pct = df_raw[feat_cols].isnull().mean().sort_values(ascending=False)
top_miss = missing_pct[missing_pct > 0].head(20)
if len(top_miss):
    axes[1].barh(top_miss.index[::-1], top_miss.values[::-1], color='salmon')
else:
    axes[1].text(0.5, 0.5, 'No Missing Values', ha='center', va='center',
                  transform=axes[1].transAxes, fontsize=11)
axes[1].set_title('Missing Rate (top 20 features)', fontweight='bold')
axes[1].set_xlabel('Missing Ratio')

# ── Feature 분산 분포 ────────────────────────────────────────
feat_var = df_raw[feat_cols].var()
axes[2].hist(feat_var, bins=50, color='mediumseagreen', edgecolor='none')
axes[2].set_title('Feature Variance Distribution', fontweight='bold')
axes[2].set_xlabel('Variance'); axes[2].set_ylabel('Count')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/01_eda.png', bbox_inches='tight')
plt.show()

print(df_raw[CD_COLUMN].describe())
print(f'\n4σ range 요약:')
print(f'  median CD   : {med_eda:.4f} nm')
print(f'  4σ_upper    : {upper_cd:.4f} nm  (norm={upper_norm:.6f})')
print(f'  4σ_lower    : {lower_cd:.4f} nm  (norm={lower_norm:.6f})')
print(f'  4σ range %  : {range_pct:.4f} %')

## 3. 전처리

In [ ]:
# 저분산 feature 제거
feat_cols_clean = feat_var[feat_var >= 1e-6].index.tolist()
print(f'저분산 제거 후 feature 수: {len(feat_cols_clean)} (제거: {len(feat_cols)-len(feat_cols_clean)}개)')

# CD 결측 제거
df_clean = df_raw.dropna(subset=[CD_COLUMN]).copy()

# Feature imputation & scaling
X_df = df_clean[feat_cols_clean].fillna(df_clean[feat_cols_clean].median())
y    = df_clean[CD_COLUMN].values.astype(np.float32)

scaler   = RobustScaler()
X_scaled = scaler.fit_transform(X_df)
X_scaled_df = pd.DataFrame(X_scaled, columns=feat_cols_clean, index=X_df.index)

# ── 전체 기준값 계산 ──────────────────────────────────────────
OVERALL_MEDIAN_CD = float(np.median(y))

# 정규화: CD_norm = CD / median_CD  →  정규화 후 median = 1
y_norm = y / OVERALL_MEDIAN_CD

# 4σ range % = (4σ_upper - 4σ_lower) × 100
#            = (mean_norm + 4σ_norm) - (mean_norm - 4σ_norm) × 100
#            = 8 × σ(CD_norm) × 100
OVERALL_MEAN_NORM  = float(y_norm.mean())
OVERALL_SIGMA_NORM = float(y_norm.std())
BASELINE_UPPER     = OVERALL_MEAN_NORM + 4 * OVERALL_SIGMA_NORM
BASELINE_LOWER     = OVERALL_MEAN_NORM - 4 * OVERALL_SIGMA_NORM
BASELINE_4SIGMA_RANGE_PCT = (BASELINE_UPPER - BASELINE_LOWER) * 100

print(f'\n전처리 완료: X={X_scaled.shape}, y={y.shape}')
print(f'  전체 CD median            : {OVERALL_MEDIAN_CD:.4f} nm')
print(f'  정규화 후 mean(CD_norm)   : {OVERALL_MEAN_NORM:.6f}  (1에 가까울수록 정상)')
print(f'  정규화 후 σ(CD_norm)      : {OVERALL_SIGMA_NORM:.6f}')
print(f'  4σ_upper = mean+4σ (norm)  : {BASELINE_UPPER:.6f}')
print(f'  4σ_lower = mean-4σ (norm)  : {BASELINE_LOWER:.6f}')
print(f'  기준선 4σ range %          : {BASELINE_4SIGMA_RANGE_PCT:.4f} %')

## 4. Cost Function (4σ range % 기반)

```
CD_norm   = CD / median_CD                # Step 1: 정규화 (median = 1)
4σ_upper  = mean(CD_norm) + 4·σ(CD_norm) # Step 2: 4σ upper bound
4σ_lower  = mean(CD_norm) - 4·σ(CD_norm) # Step 3: 4σ lower bound
4σ range % = (upper - lower) × 100       # Step 4: = 8·σ(CD_norm) × 100
```

Cost = `α × weighted_mean(4σ range %) + β × max(4σ range %)`

In [ ]:
def compute_4sigma_range_pct(cd_values: np.ndarray, ref_median: float = None) -> float:
    """4σ range % 계산.

    Step 1: CD_norm = cd_values / ref_median  (정규화 → median=1 기준)
    Step 2: 4σ_upper = mean(CD_norm) + 4·σ(CD_norm)
            4σ_lower = mean(CD_norm) - 4·σ(CD_norm)
    Step 3: 4σ range % = (4σ_upper - 4σ_lower) × 100

    Parameters
    ----------
    cd_values  : CD 값 배열 (nm)
    ref_median : 정규화 기준 median. None이면 cd_values 자체 median 사용.
    """
    if ref_median is None:
        ref_median = float(np.median(cd_values))
    if len(np.atleast_1d(cd_values)) < 2:
        return 0.0
    cd_norm  = cd_values / ref_median          # 정규화 (median=1)
    mean_n   = cd_norm.mean()
    sigma_n  = cd_norm.std()
    upper    = mean_n + 4.0 * sigma_n          # 4σ upper  (mean + 4σ)
    lower    = mean_n - 4.0 * sigma_n          # 4σ lower  (mean - 4σ)
    return (upper - lower) * 100.0             # range as %


def compute_cluster_stats(labels: np.ndarray, cd: np.ndarray) -> dict:
    """클러스터별 CD 통계 계산 (4σ range % 기준).

    모든 클러스터에 동일한 ref_median(= OVERALL_MEDIAN_CD)을 사용하여
    클러스터 간 비교 가능한 단위로 정규화.
    """
    unique_labels = np.unique(labels)
    stats_dict, count_dict, median_dict = {}, {}, {}

    for lbl in unique_labels:
        mask = labels == lbl
        cd_cl = cd[mask]
        # 전체 median 기준으로 정규화하여 클러스터 간 단위 통일
        four_range = compute_4sigma_range_pct(cd_cl, ref_median=OVERALL_MEDIAN_CD)
        stats_dict[lbl]  = four_range
        count_dict[lbl]  = int(mask.sum())
        median_dict[lbl] = float(np.median(cd_cl))
        sigma_nm_dict[lbl] = float(np.std(cd_cl))  # nm 단위 σ (저장용)

    ranges = np.array([stats_dict[l] for l in unique_labels])
    ns     = np.array([count_dict[l] for l in unique_labels])

    return {
        'four_sigma_range_pct'   : stats_dict,               # {lbl: 4σ range %}
        'sigma_per_cluster'      : sigma_nm_dict,            # {lbl: σ (nm)}
        'median_per_cluster'     : median_dict,
        'cluster_counts'         : count_dict,
        'weighted_mean_4spct'    : float(np.average(ranges, weights=ns)),
        'unweighted_mean_4spct'  : float(ranges.mean()),
        'max_4spct'              : float(ranges.max()),
        'n_clusters'             : len(unique_labels),
        'min_count'              : int(ns.min()),
    }


def cost_function(
    labels: np.ndarray,
    cd: np.ndarray,
    alpha: float = ALPHA,
    beta: float  = BETA,
    min_count: int = MIN_CLUSTER_COUNT,
) -> float:
    """Cost = α × weighted_mean(4σ range %) + β × max(4σ range %). 제약 위반 시 inf."""
    stats = compute_cluster_stats(labels, cd)
    if stats['min_count'] < min_count:
        return float('inf')
    return alpha * stats['weighted_mean_4spct'] + beta * stats['max_4spct']


def print_cluster_stats(stats: dict, method: str = '') -> None:
    baseline = BASELINE_4SIGMA_RANGE_PCT
    imp = (baseline - stats['weighted_mean_4spct']) / baseline * 100
    cost = ALPHA * stats['weighted_mean_4spct'] + BETA * stats['max_4spct']
    print(f'\n── {method} ──')
    print(f'  클러스터 수                   : {stats["n_clusters"]}')
    print(f'  최소 클러스터 크기            : {stats["min_count"]:,}')
    print(f'  기준선 4σ range %             : {baseline:.4f} %')
    print(f'  클러스터 가중평균 4σ range %  : {stats["weighted_mean_4spct"]:.4f} % (개선: {imp:.1f}%)')
    print(f'  클러스터 최대 4σ range %      : {stats["max_4spct"]:.4f} %')
    print(f'  Cost                          : {cost:.4f}')


# 기준선 확인
baseline_check = compute_4sigma_range_pct(y, ref_median=OVERALL_MEDIAN_CD)
assert abs(baseline_check - BASELINE_4SIGMA_RANGE_PCT) < 1e-8, 'Baseline mismatch!'
print('Cost function 정의 완료.')
print(f'  4σ range % = (upper - lower) × 100  (CD_norm = CD / {OVERALL_MEDIAN_CD:.4f} nm)')
print(f'  기준선 4σ range % = {BASELINE_4SIGMA_RANGE_PCT:.4f} %')

## 5. SHAP 기반 Feature Engineering

In [ ]:
# ── 5-1. XGBoost CD 예측 모델 ──────────────────────────────────
X_tr, X_val, y_tr, y_val = train_test_split(X_scaled, y, test_size=0.2, random_state=RANDOM_STATE)

xgb_model = xgb.XGBRegressor(
    n_estimators=XGB_N_ESTIMATORS, max_depth=XGB_MAX_DEPTH,
    learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
    n_jobs=-1, random_state=RANDOM_STATE, verbosity=0,
    eval_metric='rmse', early_stopping_rounds=20,
)
xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

y_pred = xgb_model.predict(X_val)
rmse = np.sqrt(np.mean((y_pred - y_val)**2))
r2   = 1 - np.sum((y_pred - y_val)**2) / np.sum((y_val - y_val.mean())**2)
print(f'XGBoost: RMSE={rmse:.3f} nm, R²={r2:.4f}')
joblib.dump(xgb_model, f'{OUTPUT_DIR}/xgb_cd_model.pkl')

In [ ]:
# ── 5-2. SHAP 계산 & Feature 선택 ─────────────────────────────
shap_idx    = np.random.RandomState(RANDOM_STATE).choice(len(X_scaled), SHAP_SAMPLE_N, replace=False)
explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_scaled[shap_idx])

shap_importance = pd.Series(
    np.abs(shap_values).mean(axis=0), index=feat_cols_clean
).sort_values(ascending=False)

selected_features = shap_importance.head(SHAP_TOP_K).index.tolist()
X_sel = X_scaled_df[selected_features].values

print(f'SHAP Top-{SHAP_TOP_K} feature 선택 완료')
cumsum_pct = shap_importance.cumsum() / shap_importance.sum() * 100
print(f'Top-{SHAP_TOP_K}이 전체 SHAP 중요도의 {cumsum_pct.iloc[SHAP_TOP_K-1]:.1f}% 커버')

In [ ]:
# ── 5-3. SHAP 시각화 ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

top_feats = shap_importance.head(SHAP_TOP_K)
colors = cm.viridis(np.linspace(0.2, 0.8, SHAP_TOP_K))[::-1]
axes[0].barh(top_feats.index[::-1], top_feats.values[::-1], color=colors)
axes[0].set_title(f'SHAP Feature Importance (Top {SHAP_TOP_K})', fontweight='bold')
axes[0].set_xlabel('mean |SHAP value|')

axes[1].plot(range(1, len(cumsum_pct)+1), cumsum_pct.values, 'steelblue', lw=2)
axes[1].axhline(90, color='red', ls='--', lw=1, label='90%')
axes[1].axhline(95, color='orange', ls='--', lw=1, label='95%')
axes[1].axvline(SHAP_TOP_K, color='green', ls='--', lw=1.5, label=f'Top-{SHAP_TOP_K}')
axes[1].set_title('Cumulative SHAP Importance', fontweight='bold')
axes[1].set_xlabel('Number of Features'); axes[1].set_ylabel('Cumulative (%)')
axes[1].legend(); axes[1].set_xlim(1, len(feat_cols_clean))

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/02_shap_importance.png', bbox_inches='tight')
plt.show()

# Beeswarm
# selected_features는 SHAP 중요도 순으로 정렬된 상위 K개
# → 원본 feature 순서 인덱스로 슬라이싱해야 정렬 일치
sel_indices = [feat_cols_clean.index(f) for f in selected_features]
shap_exp = shap.Explanation(
    values=shap_values[:, sel_indices],
    base_values=explainer.expected_value,
    data=X_scaled[shap_idx][:, sel_indices],
    feature_names=selected_features,
)
plt.figure(figsize=(10, 7))
shap.plots.beeswarm(shap_exp, max_display=SHAP_TOP_K, show=False)
plt.title(f'SHAP Beeswarm (Top {SHAP_TOP_K})', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/03_shap_beeswarm.png', bbox_inches='tight')
plt.show()

## 6. 유틸리티 함수

In [ ]:
def merge_small_clusters(labels: np.ndarray, X: np.ndarray, min_count: int) -> np.ndarray:
    """크기 min_count 미만 클러스터를 centroid KNN으로 가장 가까운 클러스터에 반복 병합."""
    labels = labels.copy()
    for _ in range(1000):
        counts = Counter(labels)
        small  = sorted([l for l, c in counts.items() if c < min_count], key=lambda l: counts[l])
        if not small:
            break
        tgt   = small[0]
        valid = [l for l in np.unique(labels) if l != tgt]
        if not valid:
            break
        mask = labels == tgt
        centroid = X[mask].mean(axis=0, keepdims=True)
        valid_centroids = np.array([X[labels == l].mean(axis=0) for l in valid])
        nearest = valid[np.linalg.norm(valid_centroids - centroid, axis=1).argmin()]
        labels[mask] = nearest
    return labels


def relabel_sequential(labels: np.ndarray) -> np.ndarray:
    mapping = {old: new for new, old in enumerate(sorted(np.unique(labels)))}
    return np.vectorize(mapping.get)(labels)


def run_grid_search(label_fn, param_range, desc):
    """Grid search 공통 루틴. label_fn(param) → raw_labels."""
    results = []
    for param in tqdm(param_range, desc=desc):
        raw     = label_fn(param)
        merged  = merge_small_clusters(raw, X_sel, MIN_CLUSTER_COUNT)
        final   = relabel_sequential(merged)
        stats   = compute_cluster_stats(final, y)
        cost    = cost_function(final, y)
        results.append({'param': param, 'labels': final, 'stats': stats, 'cost': cost})
    return results


print('유틸리티 함수 정의 완료.')

## 7. Method 1 — Decision Tree Clustering

In [ ]:
def dt_label_fn(max_leaves):
    dt = DecisionTreeRegressor(
        max_leaf_nodes=max_leaves,
        min_samples_leaf=MIN_CLUSTER_COUNT,
        random_state=RANDOM_STATE,
    )
    dt.fit(X_sel, y)
    return relabel_sequential(dt.apply(X_sel))

dt_results = run_grid_search(dt_label_fn, DT_MAX_LEAF_RANGE, 'DT Grid Search')
dt_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ['labels', 'stats']} for r in dt_results])
dt_df['n_clusters']     = [r['stats']['n_clusters']            for r in dt_results]
dt_df['mean_4spct']     = [r['stats']['weighted_mean_4spct']   for r in dt_results]
dt_df['max_4spct']      = [r['stats']['max_4spct']             for r in dt_results]
dt_df['min_count']      = [r['stats']['min_count']             for r in dt_results]

valid_dt = dt_df[dt_df['cost'] < float('inf')]
best_dt_idx = valid_dt['cost'].idxmin() if not valid_dt.empty else dt_df['cost'].idxmin()
best_dt_labels = dt_results[best_dt_idx]['labels']
best_dt_stats  = dt_results[best_dt_idx]['stats']
best_dt_param  = dt_results[best_dt_idx]['param']

print_cluster_stats(best_dt_stats, f'Decision Tree (max_leaves={best_dt_param})')

# Grid search 시각화
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(dt_df['param'], dt_df['mean_4spct'], 'o-', color='steelblue', label='Weighted Mean 4σ%')
axes[0].axvline(best_dt_param, color='red', ls='--', lw=1.5)
axes[0].axhline(BASELINE_4SIGMA_RANGE_PCT, color='gray', ls=':', lw=1.5, label=f'Baseline {BASELINE_4SIGMA_RANGE_PCT:.2f}%')
axes[0].set_title('DT: Mean 4σ% vs max_leaves'); axes[0].set_xlabel('max_leaf_nodes'); axes[0].legend()

axes[1].plot(dt_df['param'], dt_df['max_4spct'], 'o-', color='salmon', label='Max 4σ%')
axes[1].axvline(best_dt_param, color='red', ls='--', lw=1.5)
axes[1].set_title('DT: Max 4σ% vs max_leaves'); axes[1].set_xlabel('max_leaf_nodes')

valid = dt_df[dt_df['cost'] < float('inf')]
axes[2].plot(valid['param'], valid['cost'], 'o-', color='mediumseagreen', label='Cost')
axes[2].axvline(best_dt_param, color='red', ls='--', lw=1.5, label=f'Best (leaves={best_dt_param})')
axes[2].set_title(f'DT: Cost (α·mean4σ% + β·max4σ%)'); axes[2].legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/04_dt_gridsearch.png', bbox_inches='tight')
plt.show()

## 8. Method 2 — K-Means Clustering

In [ ]:
def km_label_fn(k):
    km = MiniBatchKMeans(n_clusters=k, batch_size=10_000, n_init=5, random_state=RANDOM_STATE)
    return km.fit_predict(X_sel)

km_results = run_grid_search(km_label_fn, KMEANS_K_RANGE, 'KMeans Grid Search')
km_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ['labels', 'stats']} for r in km_results])
km_df['n_clusters'] = [r['stats']['n_clusters']            for r in km_results]
km_df['mean_4spct'] = [r['stats']['weighted_mean_4spct']   for r in km_results]
km_df['max_4spct']  = [r['stats']['max_4spct']             for r in km_results]
km_df['min_count']  = [r['stats']['min_count']             for r in km_results]

valid_km = km_df[km_df['cost'] < float('inf')]
best_km_idx = valid_km['cost'].idxmin() if not valid_km.empty else km_df['cost'].idxmin()
best_km_labels = km_results[best_km_idx]['labels']
best_km_stats  = km_results[best_km_idx]['stats']
best_km_param  = km_results[best_km_idx]['param']

print_cluster_stats(best_km_stats, f'K-Means (k={best_km_param})')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(km_df['param'], km_df['mean_4spct'], 'o-', color='steelblue', label='Weighted Mean 4σ%')
axes[0].axvline(best_km_param, color='red', ls='--', lw=1.5)
axes[0].axhline(BASELINE_4SIGMA_RANGE_PCT, color='gray', ls=':', lw=1.5, label=f'Baseline {BASELINE_4SIGMA_RANGE_PCT:.2f}%')
axes[0].set_title('KMeans: Mean 4σ% vs k'); axes[0].set_xlabel('k'); axes[0].legend()

axes[1].plot(km_df['param'], km_df['max_4spct'], 'o-', color='salmon')
axes[1].axvline(best_km_param, color='red', ls='--', lw=1.5)
axes[1].set_title('KMeans: Max 4σ% vs k'); axes[1].set_xlabel('k')

valid = km_df[km_df['cost'] < float('inf')]
axes[2].plot(valid['param'], valid['cost'], 'o-', color='mediumseagreen')
axes[2].axvline(best_km_param, color='red', ls='--', lw=1.5, label=f'Best (k={best_km_param})')
axes[2].set_title('KMeans: Cost'); axes[2].legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/05_kmeans_gridsearch.png', bbox_inches='tight')
plt.show()

## 9. Method 3 — Autoencoder + K-Means (DL)

In [ ]:
# ── 9-1. Autoencoder 아키텍처 ──────────────────────────────────
class LayoutAutoencoder(nn.Module):
    """FC Autoencoder for layout feature embedding.
    
    Encoder: input → 256 → 128 → 64 → latent
    Decoder: latent → 64 → 128 → 256 → input
    BatchNorm + LeakyReLU + Dropout for robustness.
    """
    def __init__(self, input_dim: int, latent_dim: int, dropout: float = 0.1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.LeakyReLU(0.1), nn.Dropout(dropout),
            nn.Linear(256, 128),       nn.BatchNorm1d(128), nn.LeakyReLU(0.1), nn.Dropout(dropout),
            nn.Linear(128, 64),        nn.BatchNorm1d(64),  nn.LeakyReLU(0.1),
            nn.Linear(64, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.BatchNorm1d(64),  nn.LeakyReLU(0.1),
            nn.Linear(64, 128),        nn.BatchNorm1d(128), nn.LeakyReLU(0.1), nn.Dropout(dropout),
            nn.Linear(128, 256),       nn.BatchNorm1d(256), nn.LeakyReLU(0.1), nn.Dropout(dropout),
            nn.Linear(256, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

    def encode(self, x):
        return self.encoder(x)


INPUT_DIM = X_sel.shape[1]
ae_model  = LayoutAutoencoder(INPUT_DIM, AE_LATENT_DIM).to(DEVICE)
total_params = sum(p.numel() for p in ae_model.parameters())
print(f'Autoencoder 초기화 완료')
print(f'  Input dim    : {INPUT_DIM}')
print(f'  Latent dim   : {AE_LATENT_DIM}')
print(f'  Total params : {total_params:,}')
print(ae_model)

In [ ]:
# ── 9-2. Autoencoder 학습 ──────────────────────────────────────
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

X_tensor = torch.tensor(X_sel, dtype=torch.float32)
dataset  = TensorDataset(X_tensor)
loader   = DataLoader(dataset, batch_size=AE_BATCH_SIZE, shuffle=True,
                       num_workers=0, pin_memory=(DEVICE.type == 'cuda'))

optimizer = torch.optim.Adam(ae_model.parameters(), lr=AE_LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
criterion = nn.MSELoss()

train_losses, best_loss, patience_cnt = [], float('inf'), 0

print(f'Autoencoder 학습 시작 (epochs={AE_EPOCHS}, batch={AE_BATCH_SIZE}, device={DEVICE})')
for epoch in range(1, AE_EPOCHS + 1):
    ae_model.train()
    epoch_loss = 0.0
    for (xb,) in loader:
        xb = xb.to(DEVICE)
        recon, _ = ae_model(xb)
        loss = criterion(recon, xb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        epoch_loss += loss.item() * len(xb)
    epoch_loss /= len(dataset)
    train_losses.append(epoch_loss)
    scheduler.step(epoch_loss)

    if epoch_loss < best_loss:
        best_loss    = epoch_loss
        patience_cnt = 0
        torch.save(ae_model.state_dict(), f'{OUTPUT_DIR}/ae_best.pt')
    else:
        patience_cnt += 1

    if epoch % 5 == 0 or epoch == 1:
        print(f'  Epoch {epoch:3d}/{AE_EPOCHS}  loss={epoch_loss:.6f}  lr={optimizer.param_groups[0]["lr"]:.2e}')

    if patience_cnt >= AE_PATIENCE:
        print(f'  Early stopping at epoch {epoch} (patience={AE_PATIENCE})')
        break

# 학습 곡선
plt.figure(figsize=(8, 4))
plt.plot(train_losses, 'steelblue', lw=2)
plt.title('Autoencoder Training Loss (MSE)', fontweight='bold')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/06_ae_training_loss.png', bbox_inches='tight')
plt.show()
print(f'\n최적 loss: {best_loss:.6f}')

In [ ]:
# ── 9-3. Latent 표현 추출 ─────────────────────────────────────
ae_model.load_state_dict(torch.load(f'{OUTPUT_DIR}/ae_best.pt', map_location=DEVICE))
ae_model.eval()

latent_list = []
with torch.no_grad():
    for i in range(0, len(X_tensor), AE_BATCH_SIZE * 4):
        xb = X_tensor[i:i + AE_BATCH_SIZE * 4].to(DEVICE)
        latent_list.append(ae_model.encode(xb).cpu().numpy())
X_latent = np.concatenate(latent_list, axis=0)

print(f'Latent 추출 완료: {X_latent.shape}')
print(f'  Latent 표현 통계: mean={X_latent.mean():.4f}, std={X_latent.std():.4f}')

# 재구성 오류 (품질 확인)
with torch.no_grad():
    sample_idx = np.random.choice(len(X_tensor), 5000)
    xb_sample  = X_tensor[sample_idx].to(DEVICE)
    recon, _   = ae_model(xb_sample)
    recon_err  = criterion(recon, xb_sample).item()
print(f'  재구성 MSE (5K 샘플): {recon_err:.6f}')

In [ ]:
# ── 9-4. AE Latent + K-Means Grid Search ──────────────────────
# Latent 공간에 대한 그리드 서치 (X_sel 대신 X_latent 사용)
def ae_km_label_fn(k):
    km = MiniBatchKMeans(n_clusters=k, batch_size=10_000, n_init=5, random_state=RANDOM_STATE)
    return km.fit_predict(X_latent)

# run_grid_search는 X_sel로 병합 centroid를 계산하므로, latent용 별도 래퍼 정의
def run_grid_search_latent(label_fn, param_range, desc):
    results = []
    for param in tqdm(param_range, desc=desc):
        raw    = label_fn(param)
        merged = merge_small_clusters(raw, X_latent, MIN_CLUSTER_COUNT)
        final  = relabel_sequential(merged)
        stats  = compute_cluster_stats(final, y)
        cost   = cost_function(final, y)
        results.append({'param': param, 'labels': final, 'stats': stats, 'cost': cost})
    return results

ae_results = run_grid_search_latent(ae_km_label_fn, AE_K_RANGE, 'AE+KMeans Grid Search')
ae_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ['labels', 'stats']} for r in ae_results])
ae_df['n_clusters'] = [r['stats']['n_clusters']           for r in ae_results]
ae_df['mean_4spct'] = [r['stats']['weighted_mean_4spct']  for r in ae_results]
ae_df['max_4spct']  = [r['stats']['max_4spct']            for r in ae_results]
ae_df['min_count']  = [r['stats']['min_count']            for r in ae_results]

valid_ae = ae_df[ae_df['cost'] < float('inf')]
best_ae_idx = valid_ae['cost'].idxmin() if not valid_ae.empty else ae_df['cost'].idxmin()
best_ae_labels = ae_results[best_ae_idx]['labels']
best_ae_stats  = ae_results[best_ae_idx]['stats']
best_ae_param  = ae_results[best_ae_idx]['param']

print_cluster_stats(best_ae_stats, f'Autoencoder + K-Means (k={best_ae_param})')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(ae_df['param'], ae_df['mean_4spct'], 'o-', color='steelblue', label='Weighted Mean 4σ%')
axes[0].axvline(best_ae_param, color='red', ls='--', lw=1.5)
axes[0].axhline(BASELINE_4SIGMA_RANGE_PCT, color='gray', ls=':', lw=1.5, label=f'Baseline {BASELINE_4SIGMA_RANGE_PCT:.2f}%')
axes[0].set_title('AE+KMeans: Mean 4σ% vs k'); axes[0].set_xlabel('k'); axes[0].legend()

axes[1].plot(ae_df['param'], ae_df['max_4spct'], 'o-', color='salmon')
axes[1].axvline(best_ae_param, color='red', ls='--', lw=1.5)
axes[1].set_title('AE+KMeans: Max 4σ% vs k')

valid = ae_df[ae_df['cost'] < float('inf')]
axes[2].plot(valid['param'], valid['cost'], 'o-', color='mediumpurple')
axes[2].axvline(best_ae_param, color='red', ls='--', lw=1.5, label=f'Best (k={best_ae_param})')
axes[2].set_title('AE+KMeans: Cost'); axes[2].legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/07_ae_gridsearch.png', bbox_inches='tight')
plt.show()

## 10. 방법 비교 & 최종 선택

In [ ]:
dt_cost = cost_function(best_dt_labels, y)
km_cost = cost_function(best_km_labels, y)
ae_cost = cost_function(best_ae_labels, y)

comparison = pd.DataFrame([
    {
        'Method'         : 'Decision Tree',
        'Param'          : f'max_leaves={best_dt_param}',
        'n_clusters'     : best_dt_stats['n_clusters'],
        'min_count'      : best_dt_stats['min_count'],
        'mean_4σ% (↓)'   : round(best_dt_stats['weighted_mean_4spct'], 4),
        'max_4σ% (↓)'    : round(best_dt_stats['max_4spct'], 4),
        'Cost (↓)'       : round(dt_cost, 4) if dt_cost < float('inf') else 'INFEASIBLE',
    },
    {
        'Method'         : 'K-Means',
        'Param'          : f'k={best_km_param}',
        'n_clusters'     : best_km_stats['n_clusters'],
        'min_count'      : best_km_stats['min_count'],
        'mean_4σ% (↓)'   : round(best_km_stats['weighted_mean_4spct'], 4),
        'max_4σ% (↓)'    : round(best_km_stats['max_4spct'], 4),
        'Cost (↓)'       : round(km_cost, 4) if km_cost < float('inf') else 'INFEASIBLE',
    },
    {
        'Method'         : 'Autoencoder+KMeans (DL)',
        'Param'          : f'latent={AE_LATENT_DIM}, k={best_ae_param}',
        'n_clusters'     : best_ae_stats['n_clusters'],
        'min_count'      : best_ae_stats['min_count'],
        'mean_4σ% (↓)'   : round(best_ae_stats['weighted_mean_4spct'], 4),
        'max_4σ% (↓)'    : round(best_ae_stats['max_4spct'], 4),
        'Cost (↓)'       : round(ae_cost, 4) if ae_cost < float('inf') else 'INFEASIBLE',
    },
])
print('방법 비교 (4σ% 기준):')
display(comparison)

# 최종 선택
feasible = [(best_dt_labels, best_dt_stats, 'Decision Tree', dt_cost),
            (best_km_labels, best_km_stats, 'K-Means',       km_cost),
            (best_ae_labels, best_ae_stats, 'Autoencoder+KMeans (DL)', ae_cost)]
feasible = [(l, s, n, c) for l, s, n, c in feasible if c < float('inf')]
FINAL_LABELS, FINAL_STATS, FINAL_METHOD, FINAL_COST = min(feasible, key=lambda x: x[3])

print(f'\n최종 선택: {FINAL_METHOD}  (Cost={FINAL_COST:.4f})')

## 11. CD Median Alignment 효과 분석

> 각 클러스터의 CD median을 **전체 median CD**로 이동시켰을 때  
> (OPC recipe로 클러스터별 systematic offset을 보정했다고 가정)  
> 전체 4σ%가 얼마나 개선되는지 정량 분석

In [ ]:
def compute_median_aligned_4sigma_range(
    labels: np.ndarray,
    cd: np.ndarray,
    overall_median: float = OVERALL_MEDIAN_CD,
) -> dict:
    """클러스터별 CD median → overall_median으로 shift 후 전체 4σ range % 계산.

    Shift: CD_aligned_i = CD_i - cluster_median_i + overall_median
    → 클러스터 내 σ는 불변, 클러스터 간 위치(median) 차이만 제거
    → OPC recipe로 systematic offset 보정 효과 시뮬레이션
    """
    aligned_cd = cd.copy()
    offsets = {}
    for lbl in np.unique(labels):
        mask   = labels == lbl
        cl_med = float(np.median(cd[mask]))
        shift  = overall_median - cl_med
        aligned_cd[mask] += shift
        offsets[int(lbl)] = round(shift, 4)

    before_4spct  = compute_4sigma_range_pct(cd,         ref_median=overall_median)
    aligned_4spct = compute_4sigma_range_pct(aligned_cd, ref_median=overall_median)
    imp_abs       = before_4spct - aligned_4spct
    imp_rel       = imp_abs / before_4spct * 100

    return {
        'aligned_cd'       : aligned_cd,
        'before_4spct'     : before_4spct,
        'aligned_4spct'    : aligned_4spct,
        'improvement_abs'  : imp_abs,
        'improvement_rel'  : imp_rel,
        'cluster_offsets'  : offsets,
        'n_clusters'       : len(offsets),
    }


align_dt    = compute_median_aligned_4sigma_range(best_dt_labels, y)
align_km    = compute_median_aligned_4sigma_range(best_km_labels, y)
align_ae    = compute_median_aligned_4sigma_range(best_ae_labels, y)
align_final = compute_median_aligned_4sigma_range(FINAL_LABELS, y)

print('=' * 65)
print('  CD Median Alignment 효과 분석')
print('  (클러스터별 median → overall median으로 shift)')
print('=' * 65)
print(f'  전체 CD median                    : {OVERALL_MEDIAN_CD:.4f} nm')
print(f'  Alignment 전 4σ range %           : {BASELINE_4SIGMA_RANGE_PCT:.4f} %')
print()
for name, res in [('Decision Tree', align_dt), ('K-Means', align_km), ('AE+KMeans (DL)', align_ae)]:
    print(f'  [{name}]')
    print(f'    Alignment 후 4σ range %       : {res["aligned_4spct"]:.4f} %')
    print(f'    개선량 (abs)                  : {res["improvement_abs"]:.4f} % point')
    print(f'    상대 개선율                   : {res["improvement_rel"]:.2f} %')
    print()

In [ ]:
# ── 11-1. Alignment 전후 4σ% 비교 시각화 ─────────────────────
methods_align = ['Decision Tree', 'K-Means', 'AE+KMeans (DL)']
results_align = [align_dt, align_km, align_ae]

before_vals  = [BASELINE_4SIGMA_RANGE_PCT] * 3
after_vals   = [r['aligned_4spct']   for r in results_align]
imp_abs_vals = [r['improvement_abs'] for r in results_align]

x = np.arange(len(methods_align))
w = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

bars_before = axes[0].bar(x - w/2, before_vals, w, label='Before Alignment (baseline)', color='#aec7e8', edgecolor='k', lw=0.5)
bars_after  = axes[0].bar(x + w/2, after_vals,  w, label='After Alignment',             color='#2ca02c', edgecolor='k', lw=0.5, alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(methods_align, rotation=10, ha='right')
axes[0].set_ylabel('4σ range % = (upper − lower) × 100')
axes[0].set_title('4σ% Before vs After Median Alignment', fontweight='bold')
axes[0].legend()
for bar, val in zip(bars_before, before_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.01, f'{val:.3f}%', ha='center', va='bottom', fontsize=9)
for bar, val in zip(bars_after, after_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.01, f'{val:.3f}%', ha='center', va='bottom', fontsize=9, color='darkgreen')

bar_colors = ['#1f77b4', '#ff7f0e', '#9467bd']
bars_imp = axes[1].bar(methods_align, imp_abs_vals, color=bar_colors, edgecolor='k', lw=0.5, alpha=0.85)
axes[1].set_ylabel('4σ% Improvement (% point)')
axes[1].set_title('Improvement by Median Alignment\n(↑ 클수록 systematic offset 보정 효과 큼)', fontweight='bold')
axes[1].tick_params(axis='x', rotation=10)
for bar, val, r in zip(bars_imp, imp_abs_vals, results_align):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.002,
                  f'{val:.4f}%p\n({r["improvement_rel"]:.1f}% rel)',
                  ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/08_alignment_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 11-2. 최종 방법의 Alignment 상세 분석 ────────────────────
print(f'최종 방법 [{FINAL_METHOD}] Alignment 상세 분석')

# Before/After CD 분포 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 전체 CD 분포 비교
bins = np.linspace(y.min(), y.max(), 150)
axes[0].hist(y, bins=bins, alpha=0.5, color='steelblue', label=f'Before  (4σ%={BASELINE_4SIGMA_RANGE_PCT:.3f}%)', density=True)
axes[0].hist(align_final['aligned_cd'], bins=bins, alpha=0.5, color='seagreen',
             label=f'After   (4σ%={align_final["aligned_4spct"]:.3f}%)', density=True)
axes[0].axvline(OVERALL_MEDIAN_CD, color='red', ls='--', lw=1.5, label=f'Overall Median={OVERALL_MEDIAN_CD:.2f}nm')
axes[0].set_title('CD Distribution: Before vs After Alignment', fontweight='bold')
axes[0].set_xlabel('CD (nm)'); axes[0].set_ylabel('Density'); axes[0].legend()

# 클러스터별 shift 크기
offsets     = align_final['cluster_offsets']
offset_vals = list(offsets.values())
cluster_ids = list(offsets.keys())
bar_c = ['seagreen' if v > 0 else 'salmon' for v in offset_vals]
axes[1].bar([str(c) for c in cluster_ids], offset_vals, color=bar_c, edgecolor='k', lw=0.3, alpha=0.85)
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_title('Cluster Median Shift Amount (nm)\n(green: +shift, red: -shift)', fontweight='bold')
axes[1].set_xlabel('Cluster'); axes[1].set_ylabel('Shift (nm)')
if len(cluster_ids) > 20:
    axes[1].tick_params(axis='x', labelsize=7, rotation=45)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/09_alignment_detail.png', bbox_inches='tight')
plt.show()

print(f'\n  Alignment 전 4σ%  : {align_final["before_4spct"]:.4f} %')
print(f'  Alignment 후 4σ%  : {align_final["aligned_4spct"]:.4f} %')
print(f'  개선량            : {align_final["improvement_abs"]:.4f} % point')
print(f'  상대 개선율       : {align_final["improvement_rel"]:.2f} %')

In [ ]:
# ── 11-3. 클러스터별 Before/After σ & 4σ% 비교 ───────────────
n_cls = FINAL_STATS['n_clusters']

cluster_ids_sorted = sorted(FINAL_STATS['cluster_counts'].keys())
before_sigma_per  = [FINAL_STATS['sigma_per_cluster'][c] for c in cluster_ids_sorted]  # nm σ
before_4spct_per  = [FINAL_STATS['four_sigma_range_pct'][c]    for c in cluster_ids_sorted]
cluster_counts_   = [FINAL_STATS['cluster_counts'][c]    for c in cluster_ids_sorted]
cluster_medians_  = [FINAL_STATS['median_per_cluster'][c] for c in cluster_ids_sorted]

# After alignment: 각 클러스터 내부 분산은 동일 (shift만 했으므로)
# → 클러스터 내 σ는 변하지 않음. 전체 4σ%만 변화.
# 여기서는 클러스터별 CD median offset을 시각화하여 보정 필요량 표시

fig, axes = plt.subplots(2, 1, figsize=(max(12, n_cls * 0.7), 10))

x_pos = np.arange(len(cluster_ids_sorted))
bar_c = cm.RdYlGn_r(np.array(before_4spct_per) / max(before_4spct_per))
axes[0].bar(x_pos, before_4spct_per, color=bar_c, edgecolor='white', lw=0.4)
axes[0].axhline(FINAL_STATS['weighted_mean_4spct'], color='blue',  ls='--', lw=2,
                label=f'Weighted Mean 4σ% = {FINAL_STATS["weighted_mean_4spct"]:.3f}%')
axes[0].axhline(BASELINE_4SIGMA_RANGE_PCT, color='gray', ls=':',  lw=2,
                label=f'Baseline 4σ% = {BASELINE_4SIGMA_RANGE_PCT:.3f}%')
axes[0].axhline(FINAL_STATS['max_4spct'], color='red', ls='-.', lw=2,
                label=f'Max 4σ% = {FINAL_STATS["max_4spct"]:.3f}%')
axes[0].set_title(f'Per-Cluster 4σ% [{FINAL_METHOD}]', fontweight='bold')
axes[0].set_ylabel('4σ% = 4σ/median_CD × 100 (%)'); axes[0].legend()
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels([f'C{c}\n(n={cluster_counts_[i]:,})' for i, c in enumerate(cluster_ids_sorted)],
                         fontsize=8, rotation=45, ha='right')

# Cluster median vs overall median (offset 크기 = alignment에서 보정해야 할 양)
offsets_nm = [m - OVERALL_MEDIAN_CD for m in cluster_medians_]
bar_c2 = ['seagreen' if v >= 0 else 'salmon' for v in offsets_nm]
axes[1].bar(x_pos, offsets_nm, color=bar_c2, edgecolor='white', lw=0.4, alpha=0.85)
axes[1].axhline(0, color='black', lw=1)
axes[1].set_title('Cluster CD Median Offset from Overall Median (= Alignment Shift)', fontweight='bold')
axes[1].set_ylabel('Offset (nm) = Cluster Median − Overall Median')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels([f'C{c}' for c in cluster_ids_sorted], fontsize=8, rotation=45, ha='right')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/10_per_cluster_4spct.png', bbox_inches='tight')
plt.show()

## 12. 전체 방법 통합 비교

In [ ]:
# 3가지 방법 + baseline 종합 비교
all_methods = {
    'Baseline'           : {'4spct_before': BASELINE_4SIGMA_RANGE_PCT, '4spct_after': BASELINE_4SIGMA_RANGE_PCT,
                             'mean_4spct': BASELINE_4SIGMA_RANGE_PCT, 'max_4spct': BASELINE_4SIGMA_RANGE_PCT,
                             'n_clusters': 1, 'cost': ALPHA*BASELINE_4SIGMA_RANGE_PCT + BETA*BASELINE_4SIGMA_RANGE_PCT},
    'Decision Tree'      : {'4spct_before': BASELINE_4SIGMA_RANGE_PCT, '4spct_after': align_dt['aligned_4spct'],
                             'mean_4spct': best_dt_stats['weighted_mean_4spct'],
                             'max_4spct': best_dt_stats['max_4spct'],
                             'n_clusters': best_dt_stats['n_clusters'], 'cost': dt_cost},
    'K-Means'            : {'4spct_before': BASELINE_4SIGMA_RANGE_PCT, '4spct_after': align_km['aligned_4spct'],
                             'mean_4spct': best_km_stats['weighted_mean_4spct'],
                             'max_4spct': best_km_stats['max_4spct'],
                             'n_clusters': best_km_stats['n_clusters'], 'cost': km_cost},
    'AE+KMeans (DL)'     : {'4spct_before': BASELINE_4SIGMA_RANGE_PCT, '4spct_after': align_ae['aligned_4spct'],
                             'mean_4spct': best_ae_stats['weighted_mean_4spct'],
                             'max_4spct': best_ae_stats['max_4spct'],
                             'n_clusters': best_ae_stats['n_clusters'], 'cost': ae_cost},
}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
names = list(all_methods.keys())
colors_m = ['#7f7f7f', '#1f77b4', '#ff7f0e', '#9467bd']
best_marker = FINAL_METHOD

# Mean 4σ% per cluster vs baseline
mean_vals = [v['mean_4spct'] for v in all_methods.values()]
bars0 = axes[0].bar(names, mean_vals, color=colors_m, edgecolor='k', lw=0.5, alpha=0.85)
axes[0].set_title('Cluster Mean 4σ% (within cluster)', fontweight='bold')
axes[0].set_ylabel('4σ% (%)')
axes[0].tick_params(axis='x', rotation=15)
for bar, val in zip(bars0, mean_vals):
    axes[0].text(bar.get_x()+bar.get_width()/2, val+0.002, f'{val:.3f}%', ha='center', va='bottom', fontsize=9)

# After alignment 4σ%
after_vals2 = [v['4spct_after'] for v in all_methods.values()]
bars1 = axes[1].bar(names, after_vals2, color=colors_m, edgecolor='k', lw=0.5, alpha=0.85)
axes[1].set_title('Overall 4σ% After Median Alignment', fontweight='bold')
axes[1].set_ylabel('4σ% (%)')
axes[1].tick_params(axis='x', rotation=15)
for bar, val in zip(bars1, after_vals2):
    axes[1].text(bar.get_x()+bar.get_width()/2, val+0.002, f'{val:.3f}%', ha='center', va='bottom', fontsize=9)

# Cost
costs_v = [v['cost'] if v['cost'] < float('inf') else 0 for v in all_methods.values()]
bars2 = axes[2].bar(names, costs_v, color=colors_m, edgecolor='k', lw=0.5, alpha=0.85)
axes[2].set_title(f'Cost = {ALPHA}·mean(4σ%) + {BETA}·max(4σ%)', fontweight='bold')
axes[2].set_ylabel('Cost')
axes[2].tick_params(axis='x', rotation=15)
for bar, val, nm in zip(bars2, costs_v, names):
    label = f'{val:.4f}' + (' ★' if nm == best_marker else '')
    axes[2].text(bar.get_x()+bar.get_width()/2, val+0.001, label, ha='center', va='bottom', fontsize=9,
                  fontweight='bold' if nm == best_marker else 'normal')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/11_full_comparison.png', bbox_inches='tight')
plt.show()

## 13. 클러스터 시각화 (최종 방법)

In [ ]:
# ── 13-1. 클러스터별 CD 분포 (Box + σ bar) ───────────────────
n_cls    = FINAL_STATS['n_clusters']
cl_ids   = sorted(FINAL_STATS['cluster_counts'].keys())
cl_data  = [y[FINAL_LABELS == c] for c in cl_ids]
cl_4spct = [compute_4sigma_range_pct(cl_data[i], ref_median=OVERALL_MEDIAN_CD) for i in range(n_cls)]
cl_size  = [len(d) for d in cl_data]
order    = np.argsort([d.mean() for d in cl_data])

fig, axes = plt.subplots(2, 1, figsize=(max(12, n_cls * 0.7), 11))

bp = axes[0].boxplot([cl_data[order[i]] for i in range(n_cls)],
                      notch=True, patch_artist=True, showfliers=False)
for patch, c in zip(bp['boxes'], cm.RdYlGn_r(np.linspace(0.1, 0.9, n_cls))):
    patch.set_facecolor(c); patch.set_alpha(0.7)
axes[0].set_xticks(range(1, n_cls+1))
axes[0].set_xticklabels([f'C{cl_ids[order[i]]}\n(n={cl_size[order[i]]:,})' for i in range(n_cls)],
                          fontsize=8, rotation=45, ha='right')
axes[0].axhline(y.mean(), color='navy', ls='--', lw=1.2, label='Overall Mean')
axes[0].axhline(OVERALL_MEDIAN_CD, color='red', ls=':', lw=1.2, label='Overall Median')
axes[0].set_title(f'{FINAL_METHOD}: CD Distribution per Cluster', fontweight='bold')
axes[0].set_ylabel('CD (nm)'); axes[0].legend()

spct_ord = [cl_4spct[order[i]] for i in range(n_cls)]
bar_c = cm.RdYlGn_r(np.array(spct_ord)/max(spct_ord))
axes[1].bar(range(n_cls), spct_ord, color=bar_c, edgecolor='white', lw=0.4)
axes[1].axhline(FINAL_STATS['weighted_mean_4spct'], color='blue', ls='--', lw=2,
                label=f'Mean 4σ% = {FINAL_STATS["weighted_mean_4spct"]:.3f}%')
axes[1].axhline(BASELINE_4SIGMA_RANGE_PCT, color='gray', ls=':', lw=2,
                label=f'Baseline 4σ% = {BASELINE_4SIGMA_RANGE_PCT:.3f}%')
axes[1].axhline(FINAL_STATS['max_4spct'], color='red', ls='-.', lw=2,
                label=f'Max 4σ% = {FINAL_STATS["max_4spct"]:.3f}%')
axes[1].set_xticks(range(n_cls))
axes[1].set_xticklabels([f'C{cl_ids[order[i]]}' for i in range(n_cls)], fontsize=8, rotation=45, ha='right')
axes[1].set_title('Per-Cluster 4σ% = 4σ/median_CD × 100', fontweight='bold')
axes[1].set_ylabel('4σ% (%)'); axes[1].legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/12_cluster_cd_4spct.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 13-2. PCA 2D 시각화 ────────────────────────────────────────
pca_n = min(20_000, len(X_sel))
pca_i = np.random.RandomState(RANDOM_STATE).choice(len(X_sel), pca_n, replace=False)
pca   = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_sel[pca_i])

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
palette = cm.tab20(np.linspace(0, 1, n_cls))
for c in range(n_cls):
    m = FINAL_LABELS[pca_i] == c
    axes[0].scatter(X_pca[m, 0], X_pca[m, 1], s=3, alpha=0.4, color=palette[c], label=f'C{c}')
axes[0].set_title(f'PCA 2D — {FINAL_METHOD}', fontweight='bold')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
if n_cls <= 20: axes[0].legend(markerscale=3, fontsize=8, ncol=2)

sc = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=y[pca_i], cmap='plasma', s=3, alpha=0.5,
                      vmin=np.percentile(y, 2), vmax=np.percentile(y, 98))
plt.colorbar(sc, ax=axes[1], label='CD (nm)')
axes[1].set_title('PCA 2D — CD Value', fontweight='bold')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/13_pca_visualization.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 13-3. Latent 공간 PCA (AE용) ──────────────────────────────
pca_lat = PCA(n_components=2, random_state=RANDOM_STATE)
X_lat_pca = pca_lat.fit_transform(X_latent[pca_i])

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
ae_n_cls = best_ae_stats['n_clusters']
palette2 = cm.tab20(np.linspace(0, 1, ae_n_cls))
for c in range(ae_n_cls):
    m = best_ae_labels[pca_i] == c
    if m.sum() > 0:
        axes[0].scatter(X_lat_pca[m, 0], X_lat_pca[m, 1], s=3, alpha=0.4, color=palette2[c], label=f'C{c}')
axes[0].set_title(f'Latent Space PCA — AE+KMeans (k={best_ae_param})', fontweight='bold')
axes[0].set_xlabel(f'PC1 ({pca_lat.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca_lat.explained_variance_ratio_[1]*100:.1f}%)')
if ae_n_cls <= 20: axes[0].legend(markerscale=3, fontsize=8, ncol=2)

sc2 = axes[1].scatter(X_lat_pca[:, 0], X_lat_pca[:, 1], c=y[pca_i], cmap='plasma', s=3, alpha=0.5,
                       vmin=np.percentile(y, 2), vmax=np.percentile(y, 98))
plt.colorbar(sc2, ax=axes[1], label='CD (nm)')
axes[1].set_title('Latent Space PCA — CD Value', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/14_latent_pca.png', bbox_inches='tight')
plt.show()

## 14. 결과 저장

In [ ]:
# 최종 레이블 + 클러스터별 통계를 원본 df에 붙이기
df_out = df_clean.copy()
df_out['cluster']          = FINAL_LABELS
df_out['cluster_sigma_nm'] = df_out['cluster'].map(FINAL_STATS['sigma_per_cluster'])  # nm σ
df_out['cluster_4spct']    = df_out['cluster'].map(FINAL_STATS['four_sigma_range_pct'])
df_out['cluster_median_cd']= df_out['cluster'].map(FINAL_STATS['median_per_cluster'])
df_out['align_shift_nm']   = OVERALL_MEDIAN_CD - df_out['cluster_median_cd']

df_out.to_parquet(f'{OUTPUT_DIR}/clustered_data.parquet', index=True)
shap_importance.to_csv(f'{OUTPUT_DIR}/shap_importance.csv', header=['mean_abs_shap'])

summary = {
    'final_method'                   : FINAL_METHOD,
    'n_clusters'                     : int(FINAL_STATS['n_clusters']),
    'min_cluster_count'              : int(FINAL_STATS['min_count']),
    'selected_features'              : selected_features,
    'overall_median_cd_nm'           : OVERALL_MEDIAN_CD,
    'cost_alpha'                     : ALPHA,
    'cost_beta'                      : BETA,
    'cost_value'                     : round(FINAL_COST, 6),
    'baseline_4sigma_pct'            : round(BASELINE_4SIGMA_RANGE_PCT, 6),
    'cluster_weighted_mean_4spct'    : round(FINAL_STATS['weighted_mean_4spct'], 6),
    'cluster_max_4spct'              : round(FINAL_STATS['max_4spct'], 6),
    'alignment_before_4spct'         : round(align_final['before_4spct'], 6),
    'alignment_after_4spct'          : round(align_final['aligned_4spct'], 6),
    'alignment_improvement_abs_ppt'  : round(align_final['improvement_abs'], 6),
    'alignment_improvement_rel_pct'  : round(align_final['improvement_rel'], 4),
    'method_comparison': {
        'Decision Tree'   : {'cost': round(dt_cost, 4) if dt_cost < float('inf') else None,
                              'mean_4spct': round(best_dt_stats['weighted_mean_4spct'], 4),
                              'max_4spct' : round(best_dt_stats['max_4spct'], 4),
                              'align_after_4spct': round(align_dt['aligned_4spct'], 4)},
        'K-Means'         : {'cost': round(km_cost, 4) if km_cost < float('inf') else None,
                              'mean_4spct': round(best_km_stats['weighted_mean_4spct'], 4),
                              'max_4spct' : round(best_km_stats['max_4spct'], 4),
                              'align_after_4spct': round(align_km['aligned_4spct'], 4)},
        'AE+KMeans (DL)'  : {'cost': round(ae_cost, 4) if ae_cost < float('inf') else None,
                              'mean_4spct': round(best_ae_stats['weighted_mean_4spct'], 4),
                              'max_4spct' : round(best_ae_stats['max_4spct'], 4),
                              'align_after_4spct': round(align_ae['aligned_4spct'], 4)},
    },
    'config': {
        'MIN_CLUSTER_COUNT': MIN_CLUSTER_COUNT,
        'SHAP_TOP_K'       : SHAP_TOP_K,
        'AE_LATENT_DIM'    : AE_LATENT_DIM,
        'AE_EPOCHS'        : AE_EPOCHS,
        'RANDOM_STATE'     : RANDOM_STATE,
    },
}
with open(f'{OUTPUT_DIR}/clustering_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print('결과 저장 완료:')
for f in sorted(Path(OUTPUT_DIR).iterdir()):
    print(f'  {f.name}')

In [ ]:
# ── 최종 요약 출력 ─────────────────────────────────────────────
print('=' * 68)
print('  In-House Layout Clustering — 최종 결과 요약')
print('=' * 68)
print(f'  CD median (전체)          : {OVERALL_MEDIAN_CD:.4f} nm')
print(f'  기준선 4σ%                 : {BASELINE_4SIGMA_RANGE_PCT:.4f} %')
print()
print(f'  [최적 방법: {FINAL_METHOD}]')
print(f'  클러스터 수                : {FINAL_STATS["n_clusters"]}')
print(f'  최소 클러스터 크기         : {FINAL_STATS["min_count"]:,} ≥ {MIN_CLUSTER_COUNT}')
print(f'  클러스터 가중 평균 4σ%     : {FINAL_STATS["weighted_mean_4spct"]:.4f} %')
print(f'  클러스터 최대 4σ%          : {FINAL_STATS["max_4spct"]:.4f} %')
print(f'  Cost                      : {FINAL_COST:.4f}')
print()
print(f'  [Median Alignment 효과]')
print(f'  Alignment 전 4σ%           : {align_final["before_4spct"]:.4f} %')
print(f'  Alignment 후 4σ%           : {align_final["aligned_4spct"]:.4f} %')
print(f'  개선량                     : {align_final["improvement_abs"]:.4f} % point')
print(f'  상대 개선율                : {align_final["improvement_rel"]:.2f} %')
print('=' * 68)